# Anthropic Economic Index — Análise do Brasil

Exploração do dataset `Anthropic/EconomicIndex` com foco no uso de IA por ocupação e setor econômico no Brasil, comparado a países de perfil similar.

**Dataset:** https://huggingface.co/datasets/Anthropic/EconomicIndex  
**Data:** Abril 2026

## 1. Setup e Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Paths
DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../outputs')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Setup concluído.')

## 2. Download do Dataset via Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

REPO_ID = 'Anthropic/EconomicIndex'
REPO_TYPE = 'dataset'

# Listar arquivos disponíveis no repositório
files = list(list_repo_files(REPO_ID, repo_type=REPO_TYPE))
print('Arquivos disponíveis:')
for f in files:
    print(' ', f)

In [ ]:
# Baixar todos os arquivos CSV/Parquet para DATA_DIR
downloaded = []
for fname in files:
    if fname.endswith(('.csv', '.parquet', '.json', '.jsonl')):
        local_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=fname,
            repo_type=REPO_TYPE,
            local_dir=DATA_DIR
        )
        downloaded.append(local_path)
        print(f'Baixado: {local_path}')

print(f'\nTotal de arquivos baixados: {len(downloaded)}')

## 3. Exploração Inicial

In [ ]:
# Carregar o arquivo principal (ajuste o nome conforme o dataset)
csv_files = list(DATA_DIR.rglob('*.csv'))
parquet_files = list(DATA_DIR.rglob('*.parquet'))

print('CSVs:', csv_files)
print('Parquets:', parquet_files)

In [ ]:
# Carrega o primeiro arquivo encontrado (ajuste se necessário)
data_file = (csv_files + parquet_files)[0]

if str(data_file).endswith('.parquet'):
    df = pd.read_parquet(data_file)
else:
    df = pd.read_csv(data_file)

print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
df.head()

In [ ]:
# Tipos e valores nulos
df.info()
print('\nValores nulos por coluna:')
print(df.isnull().sum())

In [ ]:
# Estatísticas descritivas das colunas numéricas
df.describe()

In [ ]:
# Distribuição por coluna categórica principal (ajuste 'country' ao nome real da coluna)
country_col = [c for c in df.columns if 'country' in c.lower() or 'nation' in c.lower()]
print('Colunas de país encontradas:', country_col)

if country_col:
    print(df[country_col[0]].value_counts().head(20))

## 4. Análise do Brasil vs. Países Comparáveis

In [ ]:
# Países de referência para comparação
COMPARABLES = ['Brazil', 'Argentina', 'Mexico', 'India', 'South Africa', 'Colombia']

# Ajuste o nome da coluna de país conforme necessário
COL_COUNTRY = country_col[0] if country_col else 'country'

df_comp = df[df[COL_COUNTRY].isin(COMPARABLES)].copy()
print(f'Registros nos países comparáveis: {len(df_comp)}')
df_comp[COL_COUNTRY].value_counts()

In [ ]:
# Filtro apenas Brasil
df_brazil = df[df[COL_COUNTRY] == 'Brazil'].copy()
print(f'Registros do Brasil: {len(df_brazil)}')
df_brazil.head()

In [ ]:
# Distribuição por setor/ocupação no Brasil
# Ajuste 'sector' e 'occupation' para os nomes reais das colunas
sector_col = [c for c in df.columns if 'sector' in c.lower() or 'industry' in c.lower()]
occupation_col = [c for c in df.columns if 'occupation' in c.lower() or 'job' in c.lower() or 'soc' in c.lower()]

print('Colunas de setor:', sector_col)
print('Colunas de ocupação:', occupation_col)

In [ ]:
# Top ocupações no Brasil (ajuste a coluna conforme necessário)
if occupation_col:
    top_occ_br = df_brazil[occupation_col[0]].value_counts().head(15)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    top_occ_br.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 15 Ocupações — Brasil', fontsize=14)
    ax.set_xlabel('Frequência')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'brazil_top_occupations.png', dpi=150)
    plt.show()

## 5. Visualizações Comparativas

In [ ]:
# Volume de uso por país (países comparáveis)
country_counts = df_comp[COL_COUNTRY].value_counts().reset_index()
country_counts.columns = ['country', 'count']

fig = px.bar(
    country_counts,
    x='country', y='count',
    title='Volume de uso de IA — Países Comparáveis',
    color='country',
    labels={'count': 'Número de registros', 'country': 'País'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(showlegend=False)
fig.write_html(OUTPUT_DIR / 'comparable_countries_volume.html')
fig.show()

In [ ]:
# Heatmap: país x setor (se a coluna de setor existir)
if sector_col:
    pivot = df_comp.groupby([COL_COUNTRY, sector_col[0]]).size().unstack(fill_value=0)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
    ax.set_title('Uso de IA por País e Setor', fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'heatmap_country_sector.png', dpi=150)
    plt.show()

In [ ]:
# Treemap interativo — Brasil por setor/ocupação
if sector_col and occupation_col:
    df_brazil_grp = df_brazil.groupby([sector_col[0], occupation_col[0]]).size().reset_index(name='count')
    df_brazil_grp.columns = ['sector', 'occupation', 'count']

    fig = px.treemap(
        df_brazil_grp,
        path=['sector', 'occupation'],
        values='count',
        title='Brasil — Uso de IA por Setor e Ocupação',
        color='count',
        color_continuous_scale='Blues'
    )
    fig.write_html(OUTPUT_DIR / 'brazil_treemap_sector_occupation.html')
    fig.show()

## 6. Próximos Passos

- [ ] Mapear colunas reais do dataset após a exploração inicial
- [ ] Analisar nível de automação por ocupação no Brasil
- [ ] Comparar distribuição de tarefas (augmentation vs. automation)
- [ ] Cruzar com dados do IBGE/PNAD sobre mercado de trabalho
- [ ] Exportar insights consolidados para `outputs/`